# m4 on Colab: DistilBERT and LoRA runs

[Open in Colab](https://colab.research.google.com/github/jbouniol/nlp-project/blob/keira/colab/run_m4.ipynb)

**Before anything**: *Runtime → Change runtime type → GPU* (T4 on free Colab, L4 or A100 on Colab Pro).
Each session: run the three setup cells, then **one** run cell. Every run writes to Google Drive,
`MyDrive/ledgar/`, so a cut session loses at most one epoch: run the same cell again and it resumes.

**Protocol.** Steps 1 and 2 are *search* runs: they write validation predictions only, never the test
split. The learning rate of each method is chosen on validation macro-F1 (`docs/DECISIONS.md`); only then
does step 3 predict the test split, once, for the chosen runs. Never change a hyperparameter by hand in a
cell, and never rerun a configuration after its test predictions exist.

| step | runs | where | expected time each |
|---|---|---|---|
| 1 | full fine-tuning, lr 2e-5, 3e-5, 5e-5 (seed 0, search) | free T4 or Pro | ~25-40 min |
| 2 | LoRA r=8 q/v, lr 3e-4, 5e-4, 1e-3 (seed 0, search) | free T4 or Pro | ~20-30 min |
| — | download `MyDrive/ledgar/results/search/` into the repo's `results/search/`, then choose the learning rates | Mac | — |
| 3 | `predict_test.py` for the two chosen runs (promotion to final runs) | any GPU | ~2 min |
| 4 | seeds 1 and 2 of both chosen configurations (final runs) | Pro | as steps 1-2 |
| 5 | LoRA r=16 (L2) and LoRA on every linear layer (L3), chosen LoRA lr (search, then step 3) | Pro | as step 2 |
| 6 | optional: `bert-base-uncased`, chosen full lr | Pro | ~2x step 1 |

After a final run: download `MyDrive/ledgar/results/<run>/` into the repo's `results/<run>/` (same name),
then commit it. Full fine-tuned weights stay on Drive (`MyDrive/ledgar/models/`, 260 MB each); LoRA
adapters (a few MB) are inside the run folder and are versioned.

## Setup (once per session)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/ledgar"
COMMON = f"--out-dir {DRIVE}/results --checkpoint-dir {DRIVE}/checkpoints --models-dir {DRIVE}/models"

In [ ]:
# The repository at the head of the keira branch; Colab's own torch is kept (its CUDA build matches the GPU)
!test -d /content/nlp-project || git clone -q -b keira https://github.com/jbouniol/nlp-project.git /content/nlp-project
%cd /content/nlp-project
!git pull -q && git log --oneline -1
!pip install -q transformers==5.19.0 datasets==5.1.0 huggingface-hub==1.33.0 accelerate==1.15.0 peft==0.21.2 scikit-learn==1.9.1
!pip install -q -e . --no-deps

In [ ]:
import torch

print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU"

## Step 1: full fine-tuning, learning rate on validation (search)

If a T4 runs out of memory at `max_length` 512, add `--batch-size 16 --grad-accum 2` (same effective batch).

In [ ]:
!python scripts/train.py --model bert --search --lr-full 3e-5 --seed 0 --name bert_lr3e-5_seed0 {COMMON}

In [ ]:
!python scripts/train.py --model bert --search --lr-full 2e-5 --seed 0 --name bert_lr2e-5_seed0 {COMMON}

In [ ]:
!python scripts/train.py --model bert --search --lr-full 5e-5 --seed 0 --name bert_lr5e-5_seed0 {COMMON}

## Step 2: LoRA r = 8 on q_lin, v_lin, learning rate on validation (search)

In [ ]:
!python scripts/train.py --model lora --search --lr-lora 5e-4 --seed 0 --name lora_lr5e-4_seed0 {COMMON}

In [ ]:
!python scripts/train.py --model lora --search --lr-lora 3e-4 --seed 0 --name lora_lr3e-4_seed0 {COMMON}

In [ ]:
!python scripts/train.py --model lora --search --lr-lora 1e-3 --seed 0 --name lora_lr1e-3_seed0 {COMMON}

## Where are we? (validation only)

Lists the validation scores of every run on Drive. Download `MyDrive/ledgar/results/search/` into the
repository, and the learning rates are chosen there and written in `docs/DECISIONS.md`.

In [ ]:
import glob
import json

for path in sorted(glob.glob(f"{DRIVE}/results/**/metrics.json", recursive=True)):
    m = json.load(open(path))
    print(f"{path.split('/results/')[1]:<45} val {m.get('val')} | steps {m.get('global_step')}")

## Step 3: predict the test split once, for the chosen runs

Replace the two names by the runs chosen on validation (`docs/DECISIONS.md`). The script first checks
that the saved weights give back the validation predictions, then reads the test split, once.

In [ ]:
CHOSEN_FULL = "bert_lr3e-5_seed0"  # set from docs/DECISIONS.md
CHOSEN_LORA = "lora_lr5e-4_seed0"  # set from docs/DECISIONS.md
!python scripts/predict_test.py --name {CHOSEN_FULL} --out-dir {DRIVE}/results --models-dir {DRIVE}/models
!python scripts/predict_test.py --name {CHOSEN_LORA} --out-dir {DRIVE}/results

## Step 4: seeds 1 and 2 of the chosen configurations (final runs, with test)

Set the two learning rates chosen in step 1 and 2.

In [ ]:
LR_FULL, LR_LORA = "3e-5", "5e-4"  # set from docs/DECISIONS.md
for seed in (1, 2):
    !python scripts/train.py --model bert --lr-full {LR_FULL} --seed {seed} --name bert_lr{LR_FULL}_seed{seed} {COMMON}

In [ ]:
for seed in (1, 2):
    !python scripts/train.py --model lora --lr-lora {LR_LORA} --seed {seed} --name lora_lr{LR_LORA}_seed{seed} {COMMON}

## Step 5: LoRA variants at the chosen LoRA learning rate (search, then step 3 for each)

L2: rank 16 (alpha = 2r, so the learning rate needs no retuning, S06 p19). L3: rank 8 on every linear layer.

In [ ]:
!python scripts/train.py --model lora --search --lr-lora {LR_LORA} --lora-r 16 --lora-alpha 32 --seed 0 --name lora_r16_seed0 {COMMON}

In [ ]:
!python scripts/train.py --model lora --search --lr-lora {LR_LORA} --lora-targets q_lin k_lin v_lin out_lin lin1 lin2 --seed 0 --name lora_all_seed0 {COMMON}

## Step 6 (optional): `bert-base-uncased`, the reference model (110 M parameters)

In [ ]:
!python scripts/train.py --model bert --search --model-name bert-base-uncased --lr-full {LR_FULL} --seed 0 --name bertbase_seed0 {COMMON}